# Simulation HW 1

This notebook contains my solutions to all three questions. I have used the provided helper code as a foundation and built my own simulations with different seeds, parameters, and explanations.

In [ ]:
# Getting Python ready
from scipy.stats import norm, binom as scipy_binom, poisson as scipy_poisson
import matplotlib.pyplot as plt
import numpy as np
import math
import textwrap

In [ ]:
# ============================================================
#  Helper: Bernoulli Experiment
# ============================================================
def aBernoulliExperiment(randomNumGen, bernoulliParam_p=0.5,
                         numRepeatsOfExp=1, numTrialsInAnExp=1,
                         profitLoss=[1, 0]):
    """
    Simulate a Bernoulli experiment.
    Returns a (numRepeatsOfExp x numTrialsInAnExp) array where each
    entry is profitLoss[0] with probability p, else profitLoss[1].
    """
    observation = randomNumGen.random((numRepeatsOfExp, numTrialsInAnExp))
    index_heads = np.less_equal(observation, bernoulliParam_p)
    observation[index_heads] = profitLoss[0]
    observation[np.invert(index_heads)] = profitLoss[1]
    return observation


# ============================================================
#  Helper: Binomial Experiment
# ============================================================
def aBinomialExperiment(randomNumGen, binomialParam_n=5,
                        binomialParam_p=0.5,
                        numberOfBinomialExperiments=1,
                        profitLoss_Bern=[1, 0]):
    """
    Simulate a Binomial experiment by summing Bernoulli trials.
    Returns a 1-D array of length numberOfBinomialExperiments.
    """
    bern_outcomes = aBernoulliExperiment(
        randomNumGen,
        bernoulliParam_p=binomialParam_p,
        numRepeatsOfExp=numberOfBinomialExperiments,
        numTrialsInAnExp=binomialParam_n,
        profitLoss=profitLoss_Bern
    )
    return np.sum(bern_outcomes, axis=1)


print("Helper functions loaded successfully.")

---
## Question 1

We model a biased coin (heads with probability **p = 0.2**) as a Bernoulli(0.2) random variable.

- **Part (a):** Toss the coin 20 times; plot outcome vs. trial number.
- **Part (b):** Repeat 5 times on the same figure.
- **Part (c):** Repeat 100 times; plot histogram of total number of 1s per sequence.
- **Part (d):** From (c), keep only sequences with sum ≥ 3; histogram those sums.

### Part (a) — Single run of 20 coin tosses

In [ ]:
# Parameters
p_coin   = 0.2
n_trials = 20

rng_1a = np.random.default_rng(seed=55421)   # unique seed
obs_1a = aBernoulliExperiment(rng_1a,
                               bernoulliParam_p=p_coin,
                               numRepeatsOfExp=1,
                               numTrialsInAnExp=n_trials,
                               profitLoss=[1, 0])
# obs_1a shape: (1, 20) — flatten to 1-D for plotting
seq = obs_1a[0, :]

fig, ax = plt.subplots(figsize=(10, 3))
ax.step(range(1, n_trials + 1), seq, where='post', linewidth=1.5, color='steelblue')
ax.scatter(range(1, n_trials + 1), seq, color='tomato', zorder=5, s=40)
ax.set_xlabel('Trial Number')
ax.set_ylabel('Outcome (1 = Heads, 0 = Tails)')
ax.set_title(f'Part (a): 20 coin tosses with p = {p_coin}')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Tails (0)', 'Heads (1)'])
ax.set_xlim(0.5, n_trials + 0.5)
plt.tight_layout()
plt.show()

print(f'Number of heads observed: {int(np.sum(seq))} out of {n_trials}')

**Observation (a):** The outcomes are sparse — mostly 0s with occasional 1s, which is expected for a biased coin with p = 0.2. There is no predictable pattern; 1s appear randomly but less frequently than 0s. On average we expect 0.2 × 20 = **4 heads**, though any individual run can deviate from this.

### Part (b) — Five independent runs on the same figure

In [ ]:
num_repeats_1b = 5

rng_1b = np.random.default_rng(seed=73891)
obs_1b = aBernoulliExperiment(rng_1b,
                               bernoulliParam_p=p_coin,
                               numRepeatsOfExp=num_repeats_1b,
                               numTrialsInAnExp=n_trials,
                               profitLoss=[1, 0])

fig, ax = plt.subplots(figsize=(11, 4))

colors = ['steelblue', 'darkorange', 'green', 'crimson', 'purple']

for i in range(num_repeats_1b):
    ax.step(range(1, n_trials + 1),
            obs_1b[i, :] + i * 0.05,   # tiny vertical offset so overlapping lines stay visible
            where='post', linewidth=1.4,
            color=colors[i], label=f'Repeat {i+1}')

ax.set_xlabel('Trial Number')
ax.set_ylabel('Outcome  (offset added for visibility)')
ax.set_title(f'Part (b): 5 independent repeats of 20 tosses (p = {p_coin})')
ax.set_yticks([0, 1])
ax.set_yticklabels(['~Tails (0)', '~Heads (1)'])
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

for i in range(num_repeats_1b):
    print(f'Repeat {i+1}: heads = {int(np.sum(obs_1b[i, :]))}')

**Observation (b):** Across five repeats, each sequence has a different scatter of 1s. No two sequences are identical, confirming true randomness. However, across all repeats the density of 1s is consistently low (about 20% of trials), reflecting the underlying p = 0.2. The variability across repeats highlights that with only 20 trials, the empirical frequency can differ noticeably from the theoretical probability.

### Part (c) — Histogram of total 1s over 100 repeats of 20 tosses

In [ ]:
num_repeats_1c = 100

rng_1c = np.random.default_rng(seed=19283)
obs_1c = aBernoulliExperiment(rng_1c,
                               bernoulliParam_p=p_coin,
                               numRepeatsOfExp=num_repeats_1c,
                               numTrialsInAnExp=n_trials,
                               profitLoss=[1, 0])
# obs_1c shape: (100, 20)

# Sum of 1s per sequence
sums_1c = np.sum(obs_1c, axis=1).astype(int)   # shape: (100,)

# Histogram
fig, ax = plt.subplots(figsize=(9, 4))
bins = range(0, n_trials + 2)
histVal, edges = np.histogram(sums_1c, bins=bins)
ax.bar(edges[:-1], histVal / num_repeats_1c, width=0.7, color='steelblue', edgecolor='white')
ax.set_xlabel('Total number of 1s in a sequence of 20 tosses')
ax.set_ylabel('Frequency of occurrence')
ax.set_title('Part (c): Histogram of total heads across 100 repeats')
ax.set_xticks(range(0, n_trials + 1, 2))
plt.tight_layout()
plt.show()

print(f'Sample mean of sums: {np.mean(sums_1c):.2f}   (theoretical: {n_trials * p_coin:.2f})')
print(f'Sample std  of sums: {np.std(sums_1c):.2f}   (theoretical: {np.sqrt(n_trials * p_coin * (1-p_coin)):.2f})')

**Observation (c):** The histogram approximates a **Binomial(n=20, p=0.2) PMF**. The distribution is right-skewed and peaks near k = 3–4 (which equals n·p = 4). As we are summing 20 independent Bernoulli(0.2) trials, the total number of 1s is exactly a Binomial(20, 0.2) random variable. With only 100 repeats the histogram is noisy, but the general shape already matches the theoretical PMF.

### Part (d) — Condition on sequences with sum ≥ 3

In [ ]:
# Filter: keep only rows whose sum is at least 3
mask = sums_1c >= 3
sums_1d = sums_1c[mask]

print(f'Sequences with sum ≥ 3: {len(sums_1d)} out of {num_repeats_1c}')

fig, ax = plt.subplots(figsize=(9, 4))
histVal_d, edges_d = np.histogram(sums_1d, bins=bins)
ax.bar(edges_d[:-1], histVal_d / len(sums_1d), width=0.7,
       color='darkorange', edgecolor='white')
ax.set_xlabel('Total number of 1s in the filtered sequences')
ax.set_ylabel('Frequency of occurrence (conditional)')
ax.set_title('Part (d): Histogram conditioned on sum ≥ 3')
ax.set_xticks(range(0, n_trials + 1, 2))
plt.tight_layout()
plt.show()

print(f'Conditional mean: {np.mean(sums_1d):.2f}')

**Observation (d):** After filtering, sequences with sum < 3 are removed, so the histogram is **zero for k = 0, 1, 2**. The remaining distribution is a **truncated Binomial(20, 0.2)** — specifically the conditional PMF  $P[X = k \mid X \geq 3]$ for a Binomial(20, 0.2) variable $X$. The shape above k = 3 looks similar to the original Binomial but the probabilities are re-normalised so they sum to 1 over the allowed support {3, 4, …, 20}. The conditional mean is higher than the unconditional mean of ~4, since we have excluded the low-valued sequences.

---
## Question 2 — Binomial approximation of Poisson

We pick four values of α uniformly at random from (0, 100).

The relationship is: Binomial(n, p) → Poisson(α) as n → ∞ with np = α fixed.  
We choose the **smallest reasonable n** for each α such that the Binomial histogram visually matches the Poisson PMF.

In [ ]:
# Four α values drawn uniformly from (0, 100) with a fixed seed
rng_alpha = np.random.default_rng(seed=42)
alphas_q2 = rng_alpha.uniform(0, 100, size=4)
print(f'α values chosen: {np.round(alphas_q2, 2)}')
# Result: [77.4, 43.89, 85.86, 69.74]

In [ ]:
# For each α we choose n as the smallest integer that gives a
# visually reasonable match.  A practical rule: n ≈ 20 * α gives
# p = α/n ≈ 0.05 which is small enough for the approximation.
# We use n = round(20 * α) but enforce n >= 5*α so p <= 0.2.

def choose_n_for_alpha(alpha, multiplier=20):
    return max(int(round(multiplier * alpha)), int(5 * alpha) + 1)

n_q2 = [choose_n_for_alpha(a) for a in alphas_q2]
p_q2 = [a / n for a, n in zip(alphas_q2, n_q2)]

for i, (a, n, p) in enumerate(zip(alphas_q2, n_q2, p_q2)):
    print(f'α={a:.2f}  →  n={n}, p={p:.4f}, np={n*p:.2f}')

In [ ]:
N_EXPS_Q2 = 200_000     # large enough for smooth histograms

rng_q2 = np.random.default_rng(seed=314159)

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18, 8))
plt.tight_layout(pad=3.0, w_pad=4.0, h_pad=5.0)

for col, (alpha, n_val, p_val) in enumerate(zip(alphas_q2, n_q2, p_q2)):

    # --- Poisson simulation ---
    poisson_samples = np.random.default_rng(seed=98765 + col).integers(
        low=0, high=10**9, size=1   # just to advance; use numpy poisson
    )
    poisson_samples = np.random.poisson(lam=alpha, size=N_EXPS_Q2)

    # --- Binomial simulation ---
    binom_samples = aBinomialExperiment(rng_q2,
                                        binomialParam_n=n_val,
                                        binomialParam_p=p_val,
                                        numberOfBinomialExperiments=N_EXPS_Q2,
                                        profitLoss_Bern=[1, 0])

    max_k = int(max(np.max(poisson_samples), np.max(binom_samples))) + 2
    bins_k = range(0, max_k + 1)

    h_pois, _ = np.histogram(poisson_samples, bins=bins_k)
    h_binom, _ = np.histogram(binom_samples,  bins=bins_k)

    x_vals = np.array(range(0, max_k))
    bar_w  = 0.4

    # Top row: Poisson
    axes[0][col].bar(x_vals, h_pois / N_EXPS_Q2, width=bar_w,
                     color='steelblue', label='Poisson')
    axes[0][col].set_title(f'Poisson(α={alpha:.1f})', fontsize=9)
    axes[0][col].set_xlabel('k')
    axes[0][col].set_ylabel('Frequency')

    # Bottom row: Binomial
    axes[1][col].bar(x_vals, h_binom / N_EXPS_Q2, width=bar_w,
                     color='darkorange', label=f'Binom(n={n_val})')
    axes[1][col].set_title(f'Binom(n={n_val}, p={p_val:.4f})\nnp={n_val*p_val:.2f}',
                           fontsize=9)
    axes[1][col].set_xlabel('k')
    axes[1][col].set_ylabel('Frequency')

plt.suptitle('Question 2: Poisson(α) vs. smallest reasonable Binomial(n, α/n)',
             fontsize=12, y=1.02)
plt.show()

**Observation (Q2):**  
For each of the four randomly chosen α values (≈ 77, 44, 86, 70), choosing **n ≈ 20α** (so that p = α/n ≈ 0.05, which is small) is sufficient for the Binomial histogram to closely resemble the Poisson PMF. The two histograms in each column are nearly indistinguishable, confirming the classical Poisson limit theorem: as n → ∞ with np = α fixed, Binomial(n, p) → Poisson(α).  

For larger α the distribution is wider and more symmetric (approaching Gaussian by the CLT), while for smaller α it is more right-skewed. In both cases the Binomial with the chosen n provides a visually faithful approximation.

---
## Question 3 — LLN and CLT for sums of iid Poisson RVs

We pick **two values of α** from (1, 20) and, for each, sum increasingly many iid Poisson(α) RVs.  
We look at:
- **Law of Large Numbers (LLN):** the histogram of the *normalised* sum (i.e. the sample mean) concentrates around E[X] = α.
- **Central Limit Theorem (CLT):** the histogram of the *standardised* sum approaches a bell curve (Gaussian).

In [ ]:
# Two α values chosen from (1, 20)
rng_q3_alpha = np.random.default_rng(seed=42)
rng_q3_alpha.uniform(0, 100, 4)     # consume the same 4 draws as before
alphas_q3 = rng_q3_alpha.uniform(1, 20, size=2)
print(f'α values for Q3: {np.round(alphas_q3, 2)}')
# [2.79, 19.54]

In [ ]:
# Number of iid Poisson RVs being summed — chosen to show convergence clearly
# For small α (≈2.8) we need more terms; for large α (≈19.5) fewer terms suffice
N_VALS_SMALL_ALPHA = [2, 10, 50, 200]     # for α ≈ 2.79
N_VALS_LARGE_ALPHA = [2, 5,  20, 100]     # for α ≈ 19.54

N_EXPS_Q3 = 100_000   # experiments per setting

def simulate_poisson_sum_lln_clt(alpha, n_list, n_exps, seed=0):
    """
    For each n in n_list, simulate n_exps instances of
    S_n = X_1 + ... + X_n  where X_i ~ Poisson(alpha).
    Returns (list of S_n arrays, list of normalised S_n/n arrays).
    """
    rng = np.random.default_rng(seed=seed)
    results_sum  = []
    results_norm = []
    for n in n_list:
        # Draw n_exps × n Poisson samples, then sum across axis 1
        samples = rng.poisson(lam=alpha, size=(n_exps, n))
        S_n = np.sum(samples, axis=1).astype(float)    # (n_exps,)
        results_sum.append(S_n)
        results_norm.append(S_n / n)                   # sample mean
    return results_sum, results_norm


def plot_lln_clt(alpha, n_list, sums, norms, title_prefix):
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    plt.tight_layout(pad=3.0, w_pad=4.0, h_pad=5.0)

    # Theoretical normal for overlay
    mu_Sn_list  = [alpha * n for n in n_list]
    std_Sn_list = [np.sqrt(alpha * n) for n in n_list]
    mu_norm     = alpha

    for col, n in enumerate(n_list):
        S_arr  = sums[col]
        Xb_arr = norms[col]

        # --- Top row: histogram of S_n (CLT view — standardised) ---
        mu_s  = mu_Sn_list[col]
        sig_s = std_Sn_list[col]
        Z     = (S_arr - mu_s) / sig_s   # standardise
        axes[0][col].hist(Z, bins=60, density=True,
                          color='steelblue', alpha=0.75, edgecolor='none')
        z_grid = np.linspace(-4, 4, 300)
        axes[0][col].plot(z_grid,
                          np.exp(-0.5 * z_grid**2) / np.sqrt(2 * np.pi),
                          'r-', linewidth=1.8, label='N(0,1)')
        axes[0][col].set_title(f'n = {n}  |  CLT: standardised sum', fontsize=8)
        axes[0][col].set_xlabel('(S_n - nα) / √(nα)')
        axes[0][col].set_ylabel('Density')
        axes[0][col].legend(fontsize=7)

        # --- Bottom row: histogram of S_n / n (LLN view) ---
        axes[1][col].hist(Xb_arr, bins=60, density=True,
                          color='darkorange', alpha=0.75, edgecolor='none')
        axes[1][col].axvline(mu_norm, color='red', linewidth=1.8,
                              linestyle='--', label=f'E[X]=α={alpha:.2f}')
        axes[1][col].set_title(f'n = {n}  |  LLN: sample mean S_n/n', fontsize=8)
        axes[1][col].set_xlabel('S_n / n')
        axes[1][col].set_ylabel('Density')
        axes[1][col].legend(fontsize=7)

    plt.suptitle(f'{title_prefix}  α = {alpha:.2f}  |  {N_EXPS_Q3} experiments each',
                 fontsize=11, y=1.02)
    plt.show()

print('Functions defined.')

### α ≈ 2.79 (small α)

In [ ]:
alpha1 = alphas_q3[0]
sums1, norms1 = simulate_poisson_sum_lln_clt(alpha1, N_VALS_SMALL_ALPHA,
                                              N_EXPS_Q3, seed=20001)
plot_lln_clt(alpha1, N_VALS_SMALL_ALPHA, sums1, norms1,
             'Q3 — Small α:')

**Observation (α ≈ 2.79, small):**  

**CLT (top row):** For n = 2 the standardised sum is clearly non-Gaussian — it is discrete and right-skewed because a Poisson with small α is itself right-skewed. As n increases through 10, 50, 200 the histogram progressively converges to the standard normal curve (red line), confirming the CLT.

**LLN (bottom row):** For n = 2 the sample mean is wide and takes on only a handful of discrete values. As n grows the histogram concentrates tightly around the vertical red line at E[X] = α ≈ 2.79, demonstrating the Law of Large Numbers — the sample mean converges to the population mean.

### α ≈ 19.54 (large α)

In [ ]:
alpha2 = alphas_q3[1]
sums2, norms2 = simulate_poisson_sum_lln_clt(alpha2, N_VALS_LARGE_ALPHA,
                                              N_EXPS_Q3, seed=30002)
plot_lln_clt(alpha2, N_VALS_LARGE_ALPHA, sums2, norms2,
             'Q3 — Large α:')

**Observation (α ≈ 19.54, large):**  

**CLT (top row):** Even for n = 2 the distribution is already fairly bell-shaped because a single Poisson(19.54) RV already looks approximately Gaussian (large α means the distribution has a high mean and is more symmetric). By n = 20 the standardised histogram overlaps almost perfectly with the N(0,1) curve.

**LLN (bottom row):** The concentration around the mean is visible even at n = 2 and becomes extremely tight by n = 100. This is because the variance of the sample mean is α/n = 19.54/n, so for large α the initial spread is higher, but increasing n quickly shrinks the histogram.

**Key takeaway:** For a larger α, a shorter sequence of RVs is sufficient to observe the CLT and LLN effects. For smaller α, more RVs are needed before the distribution looks Gaussian and the mean estimate stabilises.